In [ ]:
# ============================================================
# Customer Churn Analysis using PySpark
# Author: Niloofar Mastali
# ============================================================

# Install compatible PySpark version
!pip install pyspark==3.4.1
!pip install numpy==1.26.4

In [ ]:
# ─────────────────────────────────────────────
# STEP 1: Import Libraries
# ─────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, udf
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler, StandardScaler, OneHotEncoder
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd

print('✅ Libraries imported successfully!')

In [ ]:
# ─────────────────────────────────────────────
# STEP 2: Build Spark Session & Load Data
# ─────────────────────────────────────────────
spark = SparkSession.builder.appName('Customer_Churn_Prediction').getOrCreate()

# Load dataset
data = spark.read.format('csv').option('header', True).option('inferschema', True).load('dataset.csv')
data.show(4)

# Get column types
numerical_columns = [name for name, typ in data.dtypes if typ == 'double' or typ == 'int']
categorical_columns = [name for name, typ in data.dtypes if typ == 'string']

print('Numerical columns:', numerical_columns)
print('Categorical columns:', categorical_columns)

In [ ]:
# ─────────────────────────────────────────────
# STEP 3: Exploratory Data Analysis (EDA)
# ─────────────────────────────────────────────
# Numerical features overview
df = data.select(numerical_columns).toPandas()
df.head()

# Histograms
fig = plt.figure(figsize=(15, 10))
ax = fig.gca()
df.hist(ax=ax, bins=20)
plt.tight_layout()
plt.show()

# Correlation matrix
print('Correlation Matrix:')
df.corr()

In [ ]:
# ─────────────────────────────────────────────
# STEP 4: Handle Missing Values
# ─────────────────────────────────────────────
# Check missing values
print('Missing values per column:')
for column in data.columns:
    data.select(count(when(col(column).isNull(), column)).alias(column)).show()

# Fill missing values with mean
column_with_missing_values = ['TotalCharges']
imputer = Imputer(
    inputCols=column_with_missing_values,
    outputCols=column_with_missing_values
).setStrategy('mean')

imputer = imputer.fit(data)
data = imputer.transform(data)

# Verify no missing values remain
data.select(count(when(col('TotalCharges').isNull(), 'TotalCharges')).alias('TotalCharges')).show()
print('✅ Missing values handled!')

In [ ]:
# ─────────────────────────────────────────────
# STEP 5: Remove Outliers
# ─────────────────────────────────────────────
# Check outliers in tenure column
data.select('*').where(data.tenure > 100).show()

print('Before removing outlier:', data.count())
data = data.filter(data.tenure < 100)
print('After removing outlier:', data.count())
print('✅ Outliers removed!')

In [ ]:
# ─────────────────────────────────────────────
# STEP 6: Feature Engineering
# ─────────────────────────────────────────────

# Drop rows with null values in numerical columns
data = data.na.drop(subset=numerical_columns)

# --- Numerical Features ---
# Remove existing column if present
if 'numerical_features_vector' in data.columns:
    data = data.drop('numerical_features_vector')

# Assemble numerical features into vector
numerical_vector_assembler = VectorAssembler(
    inputCols=numerical_columns,
    outputCol='numerical_features_vector'
)
data = numerical_vector_assembler.transform(data)

# Scale numerical features (mean=0, std=1)
scaler = StandardScaler(
    inputCol='numerical_features_vector',
    outputCol='numerical_features_scaled',
    withStd=True,
    withMean=True
)
data = scaler.fit(data).transform(data)

# --- Categorical Features ---
# Convert string columns to numeric
categorical_columns_indexed = [name + '_indexed' for name in categorical_columns]
indexer = StringIndexer(
    inputCols=categorical_columns,
    outputCols=categorical_columns_indexed
)
data = indexer.fit(data).transform(data)

# Remove ID and target from feature list
categorical_columns_indexed.remove('customerID_indexed')
categorical_columns_indexed.remove('Churn_indexed')

# Assemble categorical features into vector
categorical_vector_assembler = VectorAssembler(
    inputCols=categorical_columns_indexed,
    outputCol='categorical_features_vector'
)
data = categorical_vector_assembler.transform(data)

# --- Final Feature Vector ---
# Combine categorical + numerical features
final_vector_assembler = VectorAssembler(
    inputCols=['categorical_features_vector', 'numerical_features_scaled'],
    outputCol='final_feature_vector'
)
data = final_vector_assembler.transform(data)
data.select(['final_feature_vector', 'Churn_indexed']).show()

print('✅ Feature engineering complete!')

In [ ]:
# ─────────────────────────────────────────────
# STEP 7: Train / Test Split & Model Training
# ─────────────────────────────────────────────
train, test = data.randomSplit([0.7, 0.3], seed=100)
print(f'Train size: {train.count()} | Test size: {test.count()}')

# Train Decision Tree
dt = DecisionTreeClassifier(
    featuresCol='final_feature_vector',
    labelCol='Churn_indexed',
    maxDepth=3
)
model = dt.fit(train)

# Evaluate on test set
evaluator = BinaryClassificationEvaluator(labelCol='Churn_indexed')

predictions_test = model.transform(test)
auc_test = evaluator.evaluate(predictions_test, {evaluator.metricName: 'areaUnderROC'})

predictions_train = model.transform(train)
auc_train = evaluator.evaluate(predictions_train, {evaluator.metricName: 'areaUnderROC'})

print(f'Train AUC: {auc_train:.4f}')
print(f'Test  AUC: {auc_test:.4f}')

In [ ]:
# ─────────────────────────────────────────────
# STEP 8: Hyperparameter Tuning (maxDepth)
# ─────────────────────────────────────────────
def evaluate_dt(mode_params):
    test_accuracies = []
    train_accuracies = []

    for maxD in mode_params:
        # Train model
        decision_tree = DecisionTreeClassifier(
            featuresCol='final_feature_vector',
            labelCol='Churn_indexed',
            maxDepth=maxD
        )
        dtModel = decision_tree.fit(train)

        # Test AUC
        evaluator = BinaryClassificationEvaluator(labelCol='Churn_indexed')
        predictions_test = dtModel.transform(test)
        auc_test = evaluator.evaluate(predictions_test, {evaluator.metricName: 'areaUnderROC'})
        test_accuracies.append(auc_test)

        # Train AUC
        predictions_train = dtModel.transform(train)
        auc_train = evaluator.evaluate(predictions_train, {evaluator.metricName: 'areaUnderROC'})
        train_accuracies.append(auc_train)

    return test_accuracies, train_accuracies


maxDepths = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
test_accs, train_accs = evaluate_dt(maxDepths)

# Visualize results
df_results = pd.DataFrame({
    'maxDepth': maxDepths,
    'trainAUC': train_accs,
    'testAUC': test_accs
})

print(df_results)
px.line(df_results, x='maxDepth', y=['trainAUC', 'testAUC'],
        title='Train vs Test AUC by maxDepth')

In [ ]:
# ─────────────────────────────────────────────
# STEP 9: Feature Importance
# ─────────────────────────────────────────────
feature_importance = model.featureImportances
scores = [score for i, score in enumerate(feature_importance)]

# Build feature names list
feature_names = categorical_columns_indexed + numerical_columns

df_importance = pd.DataFrame(scores, columns=['score'], index=feature_names)
df_importance = df_importance.sort_values('score', ascending=False)

print('Top features influencing churn:')
print(df_importance.head(10))

px.bar(df_importance, y='score',
       title='Feature Importance — Customer Churn',
       labels={'score': 'Importance Score'})

In [ ]:
# ─────────────────────────────────────────────
# STEP 10: Business Insights
# ─────────────────────────────────────────────
# Churn rate by Contract type
df_contract = data.groupby(['Contract', 'Churn']).count().toPandas()
px.bar(df_contract, x='Contract', y='count', color='Churn',
       title='Customer Churn by Contract Type',
       barmode='group')